In [ ]:
from cloudpathlib import CloudPath, S3Client
import nibabel as nib
def load_beta_session(subject_id='subj01', resolution='func1mm', session_num=5):
    nsd_base_path = CloudPath(
        's3://natural-scenes-dataset/',
        S3Client(no_sign_request=True)
    )
    filename = f'betas_session{session_num:02d}.nii.gz'  # zero-padded
    file_path = nsd_base_path / 'nsddata_betas' / 'ppdata' / subject_id / resolution / 'betas_fithrf_GLMdenoise_RR' / filename
    print(f"Loading {file_path}")
    img = nib.load(file_path.fspath)
    print(f"Loaded image shape: {img.shape}")
    return img
def load_all_sessions(subject_id='subj01', resolution='func1mm', n_sessions=40):
    all_sessions = {}
    for session_num in range(1, n_sessions+1):
        try:
            img = load_beta_session(subject_id, resolution, session_num)
            all_sessions[session_num] = img
        except Exception as e:
            print(f"Failed to load session {session_num}: {e}")
    return all_sessions
# Load all 40 sessions for subject 'subj01'
all_session_images = load_all_sessions()

In [ ]:
# load in behavioral data

# find the shared images - should be 1000 images that are shared across all subjects 
# use 73k ID and loop through to find whether there ar
# temp = masterordering(1:750*30);  #750 trials per session; all 8 subjects participated in at least the first 30 NSD scan sessions
# shared515 = [];  #1 x 515 vector of 1-indices. these indices are between 1-1000.
# for q=1:1000
#   if sum(temp==q)==3
#     shared515 = [shared515 q];
#   end
# end

import numpy as np

# Assuming masterordering is a NumPy array
temp = masterordering[:750*30]  # First 750*30 elements

shared515 = []  # Store indices that appear exactly 3 times

for q in range(1, 1001):  # Loop from 1 to 1000 inclusive
    if np.sum(temp == q) == 3:
        shared515.append(q)

# Convert to numpy array if needed
shared515 = np.array(shared515)

# filter out "CHANGEMIND" == NaN
# filter out MISSINGDATA == 1 (buttons failed to be recorded)


# determine which sessions the images appeared in (should be first, second, and third) 


# ISOLD is 0 (the image is novel) or 1 (the image is old)
# we want to add variable that says when the item was encoded (0), when it was first seen again- retrieved (1), 
# and when it was last seen again- retrieved (2) 
# variable is encoded v. retrieved
# encoded (0) is when ISOLD == 0
# retrieved (1 & 2) is when ISOLD == 1:
    # retrieved first (1) is when session # and trial # are less than
    # retrieved second (2) is when session # and trial # are greater than 

# use filtered data to create index that can be used to filter nifti files for ONLY the trials that we want 


In [ ]:
# hard to load


from cloudpathlib import CloudPath, S3Client
import os
import nibabel as nib
import gc

# --- CONFIG ---
CACHE_DIR = "/home/jovyan/cache/"
OUTPUT_ROOT = "/home/jovyan/cache/memoryNSD"  # BIDS root folder
N_SESSIONS = 40
RESOLUTION = "func1mm"

# Setup cloud path with caching
cp = CloudPath(
    's3://natural-scenes-dataset/',
    S3Client(no_sign_request=True, local_cache_dir=CACHE_DIR)
)

betas_path = cp / "nsddata_betas/ppdata"

for subj in betas_path.iterdir():
    if not subj.name.startswith("subj"):
        continue

    subj_num = int(subj.name[-2:])
    bids_subj = f"sub-{subj_num:02d}"

    print(f"\n📦 Processing {bids_subj}...")

    beta_dir = subj / f"{RESOLUTION}/betas_fithrf_GLMdenoise_RR"

    for ses in range(1, N_SESSIONS + 1):
        session_file = f"betas_session{ses:02d}.nii.gz"
        beta_file = beta_dir / session_file

        if not beta_file.exists():
            print(f"⏩ Skipping missing session: {session_file}")
            continue

        try:
            # Load with mmap to avoid memory overload
            img = nib.load(beta_file.fspath, mmap=True)

            bids_ses = f"ses-{ses:02d}"
            output_dir = os.path.join(OUTPUT_ROOT, bids_subj, bids_ses, "func")
            os.makedirs(output_dir, exist_ok=True)

            outname = f"{bids_subj}_{bids_ses}_task-nsd.nii.gz"
            outpath = os.path.join(output_dir, outname)

            nib.save(img, outpath)

            del img
            gc.collect()

            print(f"✅ Saved session {ses} for {bids_subj} to {output_dir}")

        except Exception as e:
            print(f"❌ Error saving session {ses} for {bids_subj}: {e}")